# Text2MIDI Demo
Generates MIDI from a text description using the [amaai-lab/text2midi](https://github.com/amaai-lab/text2midi) model.

**Before running:** make sure initialization is done:
```bash
bash scripts/text2midi/initialize.sh
```

In [ ]:
import sys
import os
import pickle
import re
import torch
from omegaconf import OmegaConf
from IPython.display import Audio, display

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
CONFIG_PATH = os.path.join(PROJECT_ROOT, 'configs/text2midi/config.yaml')
MODEL_REPO = os.path.join(PROJECT_ROOT, 'models/text2midi')

assert os.path.isdir(MODEL_REPO), (
    f'Model repo not found at {MODEL_REPO}.\n'
    'Run: bash scripts/text2midi/initialize.sh'
)

cfg = OmegaConf.load(CONFIG_PATH)
print(OmegaConf.to_yaml(cfg))

In [ ]:
sys.path.insert(0, MODEL_REPO)

from model.transformer_model import Transformer
from transformers import T5Tokenizer

if torch.cuda.is_available():
    device = torch.device('cuda')
    print('Device: CUDA')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Device: MPS')
else:
    device = torch.device('cpu')
    print('Device: CPU')

In [ ]:
# Load REMI tokenizer
with open(os.path.join(cfg.paths.weights_dir, 'vocab_remi.pkl'), 'rb') as f:
    r_tokenizer = pickle.load(f)
vocab_size = len(r_tokenizer)
print(f'Vocab size: {vocab_size}')

# Load model architecture config from cloned repo
repo_cfg = OmegaConf.load(os.path.join(MODEL_REPO, 'configs/config.yaml'))
m = repo_cfg.model.text2midi_model

model = Transformer(
    n_vocab=vocab_size,
    d_model=m.decoder_d_model,
    nhead=m.decoder_num_heads,
    max_len=m.decoder_max_sequence_length,
    num_decoder_layers=m.decoder_num_layers,
    dim_feedforward=m.decoder_intermediate_size,
    use_moe=m.use_moe,
    device=device,
).to(device)

model.load_state_dict(torch.load(
    os.path.join(cfg.paths.weights_dir, 'pytorch_model.bin'),
    map_location=device,
))
model.eval()

t5_tokenizer = T5Tokenizer.from_pretrained(cfg.paths.tokenizer_dir)
print('Model ready.')

In [ ]:
# --- edit caption here ---
CAPTION = 'A happy jazz piece in C major with piano and bass'
MAX_LEN = 2000
TEMPERATURE = 1.0

inputs = t5_tokenizer(CAPTION, return_tensors='pt', padding=True, truncation=True)
input_ids = inputs.input_ids.to(device)
attention_mask = inputs.attention_mask.to(device)

with torch.no_grad():
    output = model.generate(input_ids, attention_mask, max_len=MAX_LEN, temperature=TEMPERATURE)

output_list = output[0].tolist()
generated_midi = r_tokenizer.decode(output_list)

os.makedirs(cfg.paths.output_dir, exist_ok=True)
safe_name = re.sub(r'[^\w\s-]', '', CAPTION[:50]).strip().replace(' ', '_')
out_path = os.path.join(cfg.paths.output_dir, f'{safe_name}.mid')
generated_midi.dump_midi(out_path)
print(f'MIDI saved: {out_path}')

In [ ]:
# Optional: convert to WAV and play
import subprocess

SOUNDFONT = '/usr/share/soundfonts/FluidR3_GM.sf2'
wav_path = out_path.replace('.mid', '.wav')

result = subprocess.run(
    ['fluidsynth', '-ni', SOUNDFONT, out_path, '-F', wav_path, '-r', '44100'],
    capture_output=True,
)
if result.returncode == 0:
    print(f'WAV saved: {wav_path}')
    display(Audio(wav_path))
else:
    print('fluidsynth not available. Install with: sudo dnf install fluidsynth fluid-soundfont-gm')